In [ ]:
# ================================================================
# TALLER 2 - Prediccion de precios de vivienda con Red Neuronal Densa
# (Aprendizaje Profundo).  Dataset: California Housing
# ================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
sns.set_theme(style="whitegrid")
print("TensorFlow:", tf.__version__)
print("Librerias cargadas correctamente")

In [ ]:
# 1) Carga y exploracion de los datos
data = fetch_california_housing(as_frame=True)
df = data.frame.copy()
print("Forma del dataset:", df.shape)
print("Valores nulos totales:", df.isnull().sum().sum())
df.describe().T

In [ ]:
# 2) Analisis exploratorio (EDA)
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].hist(df["MedHouseVal"], bins=50, color="#c0392b")
ax[0].set_title("Distribucion del valor de la vivienda")
ax[0].set_xlabel("Valor (x100.000 USD)")
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax[1])
ax[1].set_title("Correlacion entre variables")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
sc = plt.scatter(df["Longitude"], df["Latitude"], c=df["MedHouseVal"], cmap="viridis", s=6, alpha=0.5)
plt.colorbar(sc, label="Valor (x100.000 USD)")
plt.xlabel("Longitud")
plt.ylabel("Latitud")
plt.title("Precio de vivienda segun ubicacion en California")
plt.show()

In [ ]:
# 3) Preprocesamiento y diseno de la red neuronal densa
X = df.drop(columns="MedHouseVal").values
y = df["MedHouseVal"].values
feat = list(df.drop(columns="MedHouseVal").columns)

X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.30, random_state=SEED)
X_va, X_te, y_va, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=SEED)

scaler = StandardScaler().fit(X_tr)
X_tr_s = scaler.transform(X_tr)
X_va_s = scaler.transform(X_va)
X_te_s = scaler.transform(X_te)
print("Train:", X_tr_s.shape, " Val:", X_va_s.shape, " Test:", X_te_s.shape)

def construir_modelo(n):
    m = keras.Sequential([
        layers.Input(shape=(n,)),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(32, activation="relu"),
        layers.Dense(1)
    ])
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse", metrics=["mae"])
    return m

modelo = construir_modelo(X_tr_s.shape[1])
modelo.summary()

In [ ]:
# 4) Entrenamiento con EarlyStopping
es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)
hist = modelo.fit(X_tr_s, y_tr, validation_data=(X_va_s, y_va),
                  epochs=200, batch_size=64, callbacks=[es], verbose=1)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(hist.history["loss"], label="Entrenamiento")
ax[0].plot(hist.history["val_loss"], label="Validacion")
ax[0].set_title("Perdida (MSE)")
ax[0].legend()
ax[1].plot(hist.history["mae"], label="Entrenamiento")
ax[1].plot(hist.history["val_mae"], label="Validacion")
ax[1].set_title("Error absoluto medio (MAE)")
ax[1].legend()
plt.show()

In [ ]:
# 5) Evaluacion en el conjunto de prueba
y_pred = modelo.predict(X_te_s).flatten()
mae = mean_absolute_error(y_te, y_pred)
rmse = np.sqrt(mean_squared_error(y_te, y_pred))
r2 = r2_score(y_te, y_pred)
print("MAE :", round(mae, 3), " (aprox", format(round(mae*100000), ","), "USD de error promedio)")
print("RMSE:", round(rmse, 3))
print("R2  :", round(r2, 3), " (", round(r2*100, 1), "% de varianza explicada)")

plt.figure(figsize=(6, 6))
plt.scatter(y_te, y_pred, alpha=0.3, s=10, color="#2980b9")
plt.plot([0, 5.5], [0, 5.5], "r--", label="Prediccion perfecta")
plt.xlabel("Valor real")
plt.ylabel("Valor predicho")
plt.title("Predicciones vs valores reales")
plt.legend()
plt.show()

In [ ]:
# 6) Evaluacion etica: equidad por ingreso y peso de la ubicacion
grp = pd.qcut(X_te[:, feat.index("MedInc")], 3, labels=["Ingreso bajo", "Ingreso medio", "Ingreso alto"])
err = np.abs(y_te - y_pred)
tabla = pd.DataFrame({"g": grp, "e": err}).groupby("g", observed=True)["e"].mean()
print("Error absoluto medio por nivel de ingreso de la zona:")
print(tabla)

plt.figure(figsize=(7, 4))
plt.bar(tabla.index.astype(str), tabla.values, color=["#c0392b", "#e67e22", "#27ae60"])
plt.ylabel("Error absoluto medio")
plt.title("Equidad: error del modelo por nivel de ingreso")
plt.show()

sin = [i for i in range(len(feat)) if feat[i] not in ("Latitude", "Longitude")]
sc2 = StandardScaler().fit(X_tr[:, sin])
m2 = construir_modelo(len(sin))
m2.fit(sc2.transform(X_tr[:, sin]), y_tr,
       validation_data=(sc2.transform(X_va[:, sin]), y_va),
       epochs=200, batch_size=64,
       callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)],
       verbose=0)
r2_sin = r2_score(y_te, m2.predict(sc2.transform(X_te[:, sin])).flatten())
print("R2 CON ubicacion:", round(r2, 3), " | R2 SIN ubicacion:", round(r2_sin, 3))
print("La ubicacion aporta", round((r2 - r2_sin)*100, 1), "puntos de R2")

In [ ]:
# 7) Prueba practica y conclusiones
i = 0
pred = modelo.predict(X_te_s[i:i+1])[0][0]
print("Caracteristicas de la vivienda de ejemplo:")
for n, v in zip(feat, X_te[i]):
    print("  ", n, ":", round(float(v), 2))
print("Precio PREDICHO:", format(round(pred*100000), ","), "USD")
print("Precio REAL    :", format(round(y_te[i]*100000), ","), "USD")
print("")
print("CONCLUSION: la red neuronal densa explica cerca del", round(r2*100, 1), "% de la varianza del precio.")
print("Riesgo etico: usar latitud/longitud puede discriminar por barrio (proxy socioeconomico).")